# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use a tuned Gradient Boosted Decision Trees model (HistGradientBoostingClassifier) because search-performance dynamics contain non-linear feature interactions and temporal momentum patterns that static single-day snapshots miss.
To improve predictive signal, engineered backward-looking temporal features—such as 1-day/2-day impression deltas, impression ratios, position velocity, and 3-day rolling averages—are constructed strictly using past observations to prevent temporal lookahead bias.
The model predicts a future observed-performance proxy: a decline of at least 20% in the next observed daily impression count for the same page, provided the current page has at least 5 impressions and a valid next-day observation exists.
This is used only as a decision-support outcome. It does not establish that a page needs a refresh or that a refresh would cause recovery.

In [6]:
# Imports
import os
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    accuracy_score
)

print("Imports completed.")

# Loading the same FlyRank warehouse used in the previous notebooks
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

df = dataset["train"].to_pandas()
# Keep the same working-size approach used in ML-07
df = df.head(50000).copy()
print("Dataset shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

# Basic preparation
df["report_date"] = pd.to_datetime(df["report_date"])

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# CTR is constructed from current-day information only
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)
df["ctr"] = df["ctr"].clip(0, 1)
print(df[feature_cols + ["ctr"]].describe())

Imports completed.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset shape: (50000, 30)
Date range: 2025-01-27 to 2025-02-27
       gsc_impressions    gsc_clicks  gsc_avg_position           ctr
count     50000.000000  50000.000000      50000.000000  50000.000000
mean         15.461200      0.105300         28.559778      0.007629
std          25.695012      0.451637         22.826635      0.048061
min           1.000000      0.000000          0.000000      0.000000
25%           3.000000      0.000000          9.000000      0.000000
50%           8.000000      0.000000         21.666667      0.000000
75%          18.000000      0.000000         43.000000      0.000000
max         818.000000     16.000000        141.000000      1.000000


## Future-outcome proxy

The dataset does not contain a direct "needs refresh" label.
Therefore, this notebook uses a measurable future outcome instead.
For each page, the current row is matched with its next observed daily row. A page is marked as having a future decline when its next-day impressions are at least 20% below its current impressions.
The current-day features are kept separate from the future outcome so that the model does not see the answer while predicting it.
Rows without a valid next-day observation are excluded from model evaluation because their future outcome cannot be measured honestly.

In [7]:
# Sort so future values can be constructed correctly
df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

group_cols = [
    "client_hash_id",
    "content_hash_id"
]

# Backward-Looking Lag & Rolling Features
df["prev_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(1).fillna(df["gsc_impressions"])
df["impression_delta"] = df["gsc_impressions"] - df["prev_impressions"]
df["impression_ratio"] = (df["gsc_impressions"] + 1) / (df["prev_impressions"] + 1)

df["prev_position"] = df.groupby(group_cols)["gsc_avg_position"].shift(1).fillna(df["gsc_avg_position"])
df["position_delta"] = df["gsc_avg_position"] - df["prev_position"]

df["prev2_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(2).fillna(df["prev_impressions"])
df["impression_delta_2d"] = df["gsc_impressions"] - df["prev2_impressions"]

df["rolling_3d_impressions"] = df.groupby(group_cols)["gsc_impressions"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df["impression_vs_3d_avg"] = (df["gsc_impressions"] + 1) / (df["rolling_3d_impressions"] + 1)

# Future values for the same page
df["future_date"] = (
    df.groupby(group_cols)["report_date"]
      .shift(-1)
)

df["future_impressions"] = (
    df.groupby(group_cols)["gsc_impressions"]
      .shift(-1)
)

# Only accept the immediately following calendar day
valid_future = (
    df["future_date"] ==
    df["report_date"] + pd.Timedelta(days=1)
)

df["future_decline"] = np.nan

eligible = (
    valid_future &
    (df["gsc_impressions"] >= 5)
)

df.loc[eligible, "future_decline"] = (
    df.loc[eligible, "future_impressions"]
    <= df.loc[eligible, "gsc_impressions"] * 0.80
).astype(int)

model_df = df[df["future_decline"].notna()].copy()
print("Rows with valid future outcome:", model_df["future_decline"].notna().sum())
print("Future decline rate:")
print(model_df["future_decline"].value_counts(normalize=True, dropna=True))

Rows with valid future outcome: 27422
Future decline rate:
future_decline
0.0    0.640581
1.0    0.359419
Name: proportion, dtype: float64


## 2. Split design

A time-aware split is used because the target represents a future outcome.
The model is trained on earlier observations and evaluated on later observations. This avoids using later observations to predict earlier observations.
The final 20% of the usable observations by report date form the test period.
The same test rows are used for the baseline comparison.
This is a more realistic design for a future-opportunity question than randomly mixing dates across train and test.

In [8]:
# Sort chronologically
model_df = model_df.sort_values("report_date").reset_index(drop=True)

# Use dates rather than row positions so no date appears in both sets
unique_dates = sorted(model_df["report_date"].dropna().unique())
split_date = unique_dates[int(len(unique_dates) * 0.80)]

train_df = model_df[
    model_df["report_date"] < split_date
].copy()

test_df = model_df[
    model_df["report_date"] >= split_date
].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print(
    "\nTrain dates:",
    train_df["report_date"].min(),
    "to",
    train_df["report_date"].max()
)
print(
    "Test dates:",
    test_df["report_date"].min(),
    "to",
    test_df["report_date"].max()
)
print(
    "\nTrain target rate:",
    train_df["future_decline"].mean()
)
print(
    "Test target rate:",
    test_df["future_decline"].mean()
)
print(
    "Date overlap:",
    set(train_df["report_date"]).intersection(
        set(test_df["report_date"])
    )
)

Train rows: 14726
Test rows: 12696

Train dates: 2025-01-27 00:00:00 to 2025-02-21 00:00:00
Test dates: 2025-02-22 00:00:00 to 2025-02-26 00:00:00

Train target rate: 0.40058400108651365
Test target rate: 0.31167296786389415
Date overlap: set()


### Split limitation

A time-aware split was chosen because the target represents a future observed outcome. Earlier dates are used for training and later dates are used for testing.
The same anonymized clients can occur in both periods. Therefore, this evaluation tests temporal generalization rather than generalization to completely unseen clients.
A client-held-out evaluation would be preferable for testing cross-client generalization, but the current working slice contains only a small number of clients, making that design unsuitable for this experiment.
Features used
The model uses current-day search signals and backward-looking temporal momentum features:
gsc_impressions, gsc_clicks, gsc_avg_position, ctr, impression_delta, impression_ratio, position_delta, impression_delta_2d, impression_vs_3d_avg.
The following fields are not model features:
client_hash_id because it is an identifier
content_hash_id because it is an identifier
report_date because it defines the temporal split
future_impressions because it belongs to the future outcome
future_decline because it is the target
baseline_score, reason_code, and action because these are outputs of the Week-4 baseline rather than independent predictive signals
This prevents the model from simply learning or reproducing the existing rule.

### Features used

The model uses only information available on the current day:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `ctr`

The following fields are not model features:

- `client_hash_id` because it is an identifier
- `content_hash_id` because it is an identifier
- `report_date` because it defines the temporal split
- `future_impressions` because it belongs to the future outcome
- `future_decline` because it is the target
- `baseline_score`, `reason_code`, and `action` because these are outputs of the Week-4 baseline rather than independent predictive signals

This prevents the model from simply learning or reproducing the existing rule.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
# Model features
model_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "impression_delta",
    "impression_ratio",
    "position_delta",
    "impression_delta_2d",
    "impression_vs_3d_avg"
]

X_train = train_df[model_features]
y_train = train_df["future_decline"]

X_test = test_df[model_features]
y_test = test_df["future_decline"]
print("Model features:")
print(model_features)

# Train Tuned Multi-Lag GBDT
model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.015,
    max_depth=8,
    min_samples_leaf=15,
    l2_regularization=1.5,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.50).astype(int)
print("Model trained successfully.")

# Recreate the Week-4 baseline score on the SAME test rows.
baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    baseline_test["gsc_impressions"] * 0.4
    + (1 - baseline_test["ctr"]) * 40
    + baseline_test["gsc_avg_position"] * 0.2
)

# Higher baseline score = higher opportunity ranking
baseline_rank = (
    baseline_test["baseline_score"]
    .rank(method="first", ascending=False)
)

# Model ranking
model_rank = (
    pd.Series(model_prob, index=baseline_test.index)
    .rank(method="first", ascending=False)
)

baseline_test["model_probability"] = model_prob
baseline_test["model_rank"] = model_rank
baseline_test["baseline_rank"] = baseline_rank

print(
    baseline_test[
        [
            "report_date",
            "baseline_score",
            "model_probability",
            "future_decline"
        ]
    ].head()
)

# Evaluation helpers
def precision_at_k(y_true, scores, k=20):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y"].mean()

# Baseline
baseline_auc = roc_auc_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_ap = average_precision_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_p20 = precision_at_k(
    y_test,
    baseline_test["baseline_score"],
    20
)

# Multi-Lag GBDT Model
model_auc = roc_auc_score(
    y_test,
    model_prob
)

model_ap = average_precision_score(
    y_test,
    model_prob
)

model_p20 = precision_at_k(
    y_test,
    model_prob,
    20
)

model_acc = accuracy_score(y_test, model_pred)
base_rate = y_test.mean()

results = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Tuned Multi-Lag GBDT"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Average_Precision": [
        baseline_ap,
        model_ap
    ],
    "Precision_at_20": [
        baseline_p20,
        model_p20
    ]
})

print(f"Test-set future decline base rate: {base_rate:.4f}")
print(f"Test-set future decline percentage: {base_rate * 100:.2f}%\n")
print("Model vs baseline:")
display(results)

Model features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'impression_delta', 'impression_ratio', 'position_delta', 'impression_delta_2d', 'impression_vs_3d_avg']
Model trained successfully.
      report_date  baseline_score  model_probability  future_decline
14726  2025-02-22       42.720000           0.480064             1.0
14727  2025-02-22       49.420000           0.471126             0.0
14728  2025-02-22       47.850000           0.559424             1.0
14729  2025-02-22       47.509091           0.385529             1.0
14730  2025-02-22       52.384615           0.329545             0.0
Test-set future decline base rate: 0.3117
Test-set future decline percentage: 31.17%

Model vs baseline:


,Method,ROC_AUC,Average_Precision,Precision_at_20
0,Week-4 Baseline,0.462197,0.285572,0.25
1,Tuned Multi-Lag GBDT,0.689303,0.475635,0.35


## Markdown
How to read the comparison
The main ranking metric is Precision@20 because the intended output is a small ranked review queue.
Average Precision is also reported because it evaluates the ranking across more than one cutoff.
ROC-AUC is included as a general discrimination metric.
The future-decline base rate is reported alongside these metrics. A high precision value should not be interpreted without considering how common the future-decline outcome is.
The model is considered useful only if it provides evidence of better ranking performance than the transparent Week-4 baseline. More complexity by itself is not considered an improvement.

Cell 11: Markdown
Errors and interpretation
The model can make two important types of errors:
False positives: pages ranked as likely to decline that do not show the defined future decline.
False negatives: pages that later decline but were not ranked highly.
These errors matter because the output is intended for human review. A false positive costs reviewer time, while a false negative can cause a potentially useful review candidate to be missed.
The model is therefore treated as a prioritization aid rather than an automatic refresh decision.

In [10]:
# Error analysis
error_df = test_df.copy()

error_df["model_probability"] = model_prob
error_df["model_prediction"] = model_pred

error_df["error_type"] = np.select(
    [
        (error_df["model_prediction"] == 1) &
        (error_df["future_decline"] == 0),

        (error_df["model_prediction"] == 0) &
        (error_df["future_decline"] == 1)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

print(error_df["error_type"].value_counts())

print("\nTop model-ranked rows:")
display(
    error_df.sort_values(
        "model_probability",
        ascending=False
    )[
        [
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr",
            "model_probability",
            "future_decline",
            "error_type"
        ]
    ].head(20)
)

# Compare top-20 baseline and model overlap
baseline_top20 = set(
    baseline_test.nlargest(
        20,
        "baseline_score"
    ).index
)

model_top20 = set(
    pd.Series(model_prob, index=test_df.index)
    .nlargest(20)
    .index
)

overlap = len(
    baseline_top20.intersection(model_top20)
)
print("Top-20 overlap between baseline and model:", overlap, "/ 20")
# Produce a model-ranked decision-support queue
model_queue = test_df.copy()

model_queue["model_probability"] = model_prob

model_queue = model_queue.sort_values(
    "model_probability",
    ascending=False
).reset_index(drop=True)

model_queue["rank"] = (
    np.arange(len(model_queue)) + 1
)

model_queue["decision_support"] = np.where(
    model_queue["model_probability"] >= 0.50,
    "Review",
    "Monitor"
)

display(
    model_queue[
        [
            "rank",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ctr",
            "model_probability",
            "decision_support"
        ]
    ].head(20)
)

# Save only a small metrics receipt.
os.makedirs("work/outputs", exist_ok=True)

metrics = {
    "random_seed": 42,
    "test_base_rate": float(base_rate),
    "test_accuracy": float(model_acc),
    "baseline_roc_auc": float(baseline_auc),
    "model_roc_auc": float(model_auc),
    "baseline_average_precision": float(baseline_ap),
    "model_average_precision": float(model_ap),
    "baseline_precision_at_20": float(baseline_p20),
    "model_precision_at_20": float(model_p20),
    "top20_overlap": int(overlap)
}

with open(
    "work/outputs/ml08_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print("Metrics receipt saved.")
print(json.dumps(metrics, indent=2))

error_type
Correct           8543
False Positive    2472
False Negative    1681
Name: count, dtype: int64

Top model-ranked rows:


,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,model_probability,future_decline,error_type
16856,2025-02-22,60.0,0.0,29.716667,0.0,0.663058,0.0,False Positive
21483,2025-02-24,12.0,0.0,25.166667,0.0,0.658436,1.0,Correct
21013,2025-02-24,10.0,0.0,17.100000,0.0,0.657931,1.0,Correct
21242,2025-02-24,16.0,0.0,18.562500,0.0,0.657931,0.0,False Positive
22697,2025-02-24,9.0,0.0,19.777778,0.0,0.657931,1.0,Correct
24369,2025-02-25,8.0,0.0,16.125000,0.0,0.655283,1.0,Correct
23440,2025-02-24,21.0,0.0,17.380952,0.0,0.655237,0.0,False Positive
23472,2025-02-24,27.0,0.0,16.407407,0.0,0.655237,0.0,False Positive
23144,2025-02-24,23.0,0.0,24.043478,0.0,0.654783,0.0,False Positive
17083,2025-02-22,29.0,0.0,26.068966,0.0,0.654783,0.0,False Positive


Top-20 overlap between baseline and model: 0 / 20


,rank,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,model_probability,decision_support
0,1,2025-02-22,60.0,0.0,29.716667,0.0,0.663058,Review
1,2,2025-02-24,12.0,0.0,25.166667,0.0,0.658436,Review
2,3,2025-02-24,10.0,0.0,17.100000,0.0,0.657931,Review
3,4,2025-02-24,16.0,0.0,18.562500,0.0,0.657931,Review
4,5,2025-02-24,9.0,0.0,19.777778,0.0,0.657931,Review
5,6,2025-02-25,8.0,0.0,16.125000,0.0,0.655283,Review
6,7,2025-02-24,21.0,0.0,17.380952,0.0,0.655237,Review
7,8,2025-02-24,27.0,0.0,16.407407,0.0,0.655237,Review
8,9,2025-02-24,23.0,0.0,24.043478,0.0,0.654783,Review
9,10,2025-02-22,29.0,0.0,26.068966,0.0,0.654783,Review


Metrics receipt saved.
{
  "random_seed": 42,
  "test_base_rate": 0.31167296786389415,
  "test_accuracy": 0.6728890989287964,
  "baseline_roc_auc": 0.46219748206944766,
  "model_roc_auc": 0.6893029579363904,
  "baseline_average_precision": 0.2855718831520397,
  "model_average_precision": 0.4756351933152646,
  "baseline_precision_at_20": 0.25,
  "model_precision_at_20": 0.35,
  "top20_overlap": 0
}


## Interpretation
The Tuned Multi-Lag GBDT model achieved an ROC-AUC of 0.6893 and an Average Precision of 0.4756, significantly outperforming the Week-4 baseline (0.4622 ROC-AUC and 0.2856 AP) on the held-out period. Furthermore, Precision@20 increased from 0.2500 to 0.3500.

What the model does differently
By utilizing multi-day backward rolling window features (3-day impression averages, position deltas, and impression velocity) rather than static snapshots, the model learns trend momentum without lookahead bias. This allows it to surface actionable review candidates more accurately in the top-ranked decision-support queue.

Limitations & Self-check
This experiment uses a 50,000-row working slice rather than the full warehouse. The results should be treated as directional evidence from this experiment, not as a benchmark for the complete FlyRank warehouse.
The future-decline proxy is based on the next observed daily impression count. It is not a direct human-labelled "needs refresh" outcome.
Because the same clients can appear in both temporal periods, this evaluation measures temporal generalization rather than unseen-client generalization.
The model does not establish that a page needs a refresh, that a refresh would improve performance, or that any observed movement was caused by content changes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.